# Vizualizacia segmentacie optickeho disku (OD) a poharika (OC)

Tento notebook je pripraveny ako verejna ukazka pre vysledok mojej bakalarskej prace pomocou GitHub + Google Colab.

Notebook po spusteni v sluzbe colab.research.google.com automaticky naklonuje repozitar (vahy a dataset) a spusti inferenciu mnou natrenovanej neuronovej siete

### Ako spustit inferenciu

- Prihlaste sa svojim Google uctom. Google Colab je oficialna sluzba poskytovana firmou Google.
- Zmente nastavenie `Runtime > Change runtime type`.
- Nastavte hodnotu `Runtime type` na `Python 3` a potvrdte tlacidlom `Save`.
- Odporucany krok: Pre rychlejsiu inferenciu zmente nastavenie `Runtime > Change runtime type > Hardware accelerator` na hodnotu `T4 GPU` a potvrdte tlacidlom `Save`.
- Spustenie inferencie vykonate stlacenim tlacidla `Run all`.

### Vyber testovacieho obrazka

V priecinku `data/test` su pripravene 4 ukazkove sety obrazkov (fundus snimok + maska). Obrazky boli nahodne vybrane z testovacej mnoziny a su pomenovane podla nazvu povodnyh datasetov.

Ak chcete vyskusat iny obrazok, zakomentujte aktualny riadok znakom `#` a odkomentujte prave jeden iny obrazok zmazanim znaku `#`.

In [ ]:
# ----------------- VYBER TESTOVACIEHO OBRAZKA -----------------
SELECTED_IMAGE_NAME = "ORIGA_1.jpg"
# SELECTED_IMAGE_NAME = "ORIGA_2.jpg"
# SELECTED_IMAGE_NAME = "G1020_1.jpg"
# SELECTED_IMAGE_NAME = "G1020_2.jpg"
# ---------------------------------------------------------------

In [ ]:
from pathlib import Path
import subprocess
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

GITHUB_REPO_URL = "https://github.com/marekfejda/Bakalarska_praca"
PROJECT_ROOT = Path("/content/Bakalarska_praca")

if not PROJECT_ROOT.exists():
    print(f"Klonujem repozitar: {GITHUB_REPO_URL}")
    subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, str(PROJECT_ROOT)],
        check=True
    )

os.chdir(PROJECT_ROOT)

print(f"CUDA is available: {torch.cuda.is_available()}")

## 1. Nastavenia a cesty

In [ ]:
# ----------------- NASTAVENIA -----------------
MODEL_WEIGHTS_PATH = PROJECT_ROOT / "weights" / "best_weights.pth"
TEST_IMAGE_PATH = PROJECT_ROOT / "data" / "test" / SELECTED_IMAGE_NAME
TEST_MASK_PATH = TEST_IMAGE_PATH.with_suffix(".png")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ----------------------------------------------

print("MODEL_WEIGHTS_PATH:", MODEL_WEIGHTS_PATH)
print("TEST_IMAGE_PATH:   ", TEST_IMAGE_PATH)
print("TEST_MASK_PATH:    ", TEST_MASK_PATH)
print("DEVICE:            ", DEVICE)


if not all(Path(p).exists() for p in [MODEL_WEIGHTS_PATH, TEST_IMAGE_PATH, TEST_MASK_PATH]):
    raise FileNotFoundError("Error: One or more required files are missing")

## 2. Architektura siete U-Net (s ResNet34 transfer learningom) a nacitanie vah modelu

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)
    
    def forward(self, x, skip):
        x = self.up(x)
        
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        x = F.pad(x, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, out_ch=3):
        super().__init__()
        # Stiahneme a nacitame pred-trenovane vahy (ImageNet)
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        
        # ENCODER CAST
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64, H/2
        self.pool0 = resnet.maxpool                                       # H/4
        self.enc1 = resnet.layer1                                         # 64
        self.enc2 = resnet.layer2                                         # 128
        self.enc3 = resnet.layer3                                         # 256
        self.enc4 = resnet.layer4                                         # 512
        
        # DECODER CAST
        self.up1 = Up(512, 256, 256)
        self.up2 = Up(256, 128, 128)
        self.up3 = Up(128, 64, 64)
        self.up4 = Up(64, 64, 64)
        
        self.up_final = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dev_final = DoubleConv(32, 32)
        
        self.out = nn.Conv2d(32, out_ch, 1)
    
    def forward(self, x):
        x0 = self.enc0(x)      
        p0 = self.pool0(x0)
        x1 = self.enc1(p0)     
        x2 = self.enc2(x1)     
        x3 = self.enc3(x2)     
        x4 = self.enc4(x3)     
        
        # Decoder
        d1 = self.up1(x4, x3)  
        d2 = self.up2(d1, x2)  
        d3 = self.up3(d2, x1)  
        d4 = self.up4(d3, x0)  
        
        d_final = self.up_final(d4)
        d_final = self.dev_final(d_final)
        out = self.out(d_final)
        
        return out


model = UNet(out_ch=3).to(DEVICE)

print(f"Loading model weights from: {MODEL_WEIGHTS_PATH}")
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
model.eval()

## 3. Nacitanie obrazku a generovanie predikcie

In [ ]:
print(f"Loading image: {TEST_IMAGE_PATH}")

original_image_rgb = cv2.cvtColor(cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
ground_truth_label = np.array(Image.open(TEST_MASK_PATH).convert('L'))

image_float = original_image_rgb.astype(np.float32) / 255.0
input_tensor = torch.from_numpy(image_float.transpose(2, 0, 1)[None, ...]).float().to(DEVICE)

# Inferencia
with torch.no_grad():
    logits_output = model(input_tensor)
    # Pravdepodobnostna mapa a argmax masky
    probability_map = torch.softmax(logits_output, dim=1).cpu().numpy()[0]
    raw_prediction_mask = torch.argmax(logits_output, dim=1).cpu().numpy()[0]

print("Prediction generated successfully.")

## Vypocet metrik (IoU, Dice)
Vypocitame metriky (Intersection over Union a Dice koeficient) pre opticky poharik (OC) a opticky disk (OD), aby sme mohli kvantitativne zhodnotit uspesnost modelu.

In [ ]:
def compute_metrics(pred, gt, cls):
    p = (pred == cls)
    g = (gt == cls)
    intersection = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    iou = intersection / union if union > 0 else 0.0
    dice = 2.0 * intersection / (p.sum() + g.sum()) if (p.sum() + g.sum()) > 0 else 0.0
    return iou, dice

iou_bg, dice_bg = compute_metrics(raw_prediction_mask, ground_truth_label, 0)
iou_disc, dice_disc = compute_metrics(raw_prediction_mask, ground_truth_label, 1)
iou_cup, dice_cup = compute_metrics(raw_prediction_mask, ground_truth_label, 2)

print(f"Background (Label 0) - IoU: {iou_bg:.4f}, Dice: {dice_bg:.4f}")
print(f"Optic Disc (Label 1) - IoU: {iou_disc:.4f}, Dice: {dice_disc:.4f}")
print(f"Optic Cup (Label 2)  - IoU: {iou_cup:.4f}, Dice: {dice_cup:.4f}")

Na zaklade poctu pixelov v predikovanej oblasti OC a OD sa vypocita hodnota ACDR, podla ktorej sa nalez orientacne klasifikuje ako normalny, podozrivy alebo pravdepodobne glaukomaticky.

In [ ]:
# ----------------- ODHAD ACDR A KLASIFIKACIA -----------------
# Labely:
# 0 = pozadie
# 1 = opticky disk OD
# 2 = opticka jamka OC
# --------------------------------------------------------------
prediction_mask = raw_prediction_mask
gt_mask = ground_truth_label

def calc_acdr_and_result(mask, title):
    oc_pixels = np.sum(mask == 2)
    od_pixels = np.sum((mask == 1) | (mask == 2))
    
    acdr = oc_pixels / od_pixels if od_pixels > 0 else 0.0
    
    if acdr < 0.35:
        cdr_result = "Normalny nalez - glaukom sa nepredpoklada"
    else:
        cdr_result = "Pravdepodobny glaukom"

    print(f"\n--- ODHAD CUP-TO-DISC RATIO ({title}) ---")
    print(f"Pocet pixelov OC: {oc_pixels}")
    print(f"Pocet pixelov OD: {od_pixels}")
    print(f"ACDR: {acdr:.4f}")
    print(f"Vysledok: {cdr_result}")
    
    return acdr, cdr_result

pred_acdr, pred_cdr_result = calc_acdr_and_result(prediction_mask, "PREDIKCIA")
gt_acdr, gt_cdr_result = calc_acdr_and_result(gt_mask, "GROUND TRUTH")

## 4. Renderovanie a matica matplot

In [ ]:
SEMANTIC_COLORS = {
    1: (0, 255, 0),      # GT Optic Disc
    2: (255, 0, 0)       # GT Optic Cup
}

PREDICTION_COLORS = {
    1: (255, 255, 0),    # Pred Optic Disc
    2: (0, 0, 255)       # Pred Optic Cup
}

def draw_contours(image_rgb, mask, colors):
    image_bgr = cv2.cvtColor(image_rgb.copy(), cv2.COLOR_RGB2BGR)

    for label in colors:
        binary_mask = (mask == label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        color = colors[label]
        cv2.drawContours(image_bgr, contours, -1, (color[2], color[1], color[0]), 2)

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# -------------------------------------------------
# 1. Colored GT mask
gt_col = np.zeros((ground_truth_label.shape[0], ground_truth_label.shape[1], 3), dtype=np.uint8)
gt_col[ground_truth_label == 1] = (0, 255, 0)
gt_col[ground_truth_label == 2] = (255, 0, 0)

# -------------------------------------------------
# 2. Colored predicted mask
pred_col = np.zeros((raw_prediction_mask.shape[0], raw_prediction_mask.shape[1], 3), dtype=np.uint8)
pred_col[raw_prediction_mask == 1] = (255, 255, 0)
pred_col[raw_prediction_mask == 2] = (0, 0, 255)

# -------------------------------------------------
# 3. Contours
gt_cont = draw_contours(original_image_rgb, ground_truth_label, SEMANTIC_COLORS)
pred_cont = draw_contours(original_image_rgb, raw_prediction_mask, PREDICTION_COLORS)

# -------------------------------------------------
# 4. Final overlay - first GT, then prediction
overlay = draw_contours(original_image_rgb, ground_truth_label, SEMANTIC_COLORS)
overlay = draw_contours(overlay, raw_prediction_mask, PREDICTION_COLORS)

# -------------------------------------------------
# 5. Figure 4x3
fig, axes = plt.subplots(4, 3, figsize=(15, 18))
axes = axes.ravel()

# 1
axes[0].imshow(pred_col)
axes[0].set_title("1) Prediction")
axes[0].axis("off")

# 2
axes[1].imshow(gt_col)
axes[1].set_title("2) Ground Truth")
axes[1].axis("off")

# 3
axes[2].imshow(original_image_rgb)
axes[2].set_title("3) Original Image")
axes[2].axis("off")

# 4
axes[3].imshow(probability_map[2], cmap="inferno", vmin=0, vmax=1)
axes[3].set_title("4) Cup Probability Map")
axes[3].axis("off")

# 5
axes[4].imshow(probability_map[1], cmap="inferno", vmin=0, vmax=1)
axes[4].set_title("5) Disc Probability Map")
axes[4].axis("off")

# 6
axes[5].imshow(gt_cont)
axes[5].set_title("6) Ground Truth Contours")
axes[5].axis("off")

# 7
axes[6].imshow(pred_cont)
axes[6].set_title("7) Predicted Contours")
axes[6].axis("off")

# 8
axes[7].imshow(overlay)
axes[7].set_title("8) Final Overlay")
axes[7].axis("off")

# 9
axes[8].axis("off")
axes[8].set_title("9) Legend")

legend_elements = [
    mpatches.Patch(color=np.array([0, 255, 0]) / 255.0, label="GT Optic Disc"),
    mpatches.Patch(color=np.array([255, 0, 0]) / 255.0, label="GT Optic Cup"),
    mpatches.Patch(color=np.array([255, 255, 0]) / 255.0, label="Predicted Optic Disc"),
    mpatches.Patch(color=np.array([0, 0, 255]) / 255.0, label="Predicted Optic Cup")
]

axes[8].legend(handles=legend_elements, loc="center", frameon=True)

# 10 - ACDR thresholds
axes[9].axis("off")
axes[9].set_title("10) aCDR Thresholds")

threshold_text = (
    "Normalny nalez: ACDR < 0.5\n\n"
    "Podozrivy nalez: 0.5 ≤ ACDR < 0.6\n\n"
    "Pravdepodobny glaukom: ACDR ≥ 0.6"
)

axes[9].text(0.05, 0.85, threshold_text, fontsize=12, va="top")

# 11 - evaluation
axes[10].axis("off")
axes[10].set_title("11) Prediction and Ground Truth Evaluation")

axes[10].text(0.05, 0.85, "PREDICTION", fontsize=12, fontweight="bold", va="top")
axes[10].text(0.05, 0.73, f"ACDR: {pred_acdr:.3f}", fontsize=12, va="top")
axes[10].text(0.05, 0.61, pred_cdr_result, fontsize=12, fontweight="bold", va="top")

axes[10].text(0.05, 0.39, "GROUND TRUTH", fontsize=12, fontweight="bold", va="top")
axes[10].text(0.05, 0.27, f"ACDR: {gt_acdr:.3f}", fontsize=12, va="top")
axes[10].text(0.05, 0.15, gt_cdr_result, fontsize=12, fontweight="bold", va="top")

# 12 - metrics
axes[11].axis("off")
axes[11].set_title("12) Metrics")

metrics_text = (
    f"Background (Label 0) - IoU: {iou_bg:.3f}, Dice: {dice_bg:.3f}\n\n"
    f"Optic Disc (Label 1) - IoU: {iou_disc:.3f}, Dice: {dice_disc:.3f}\n\n"
    f"Optic Cup (Label 2) - IoU: {iou_cup:.3f}, Dice: {dice_cup:.3f}"
)

axes[11].text(0.05, 0.85, metrics_text, fontsize=11, va="top")

plt.tight_layout()
plt.show()